# Collating Outputs from Multiple Experiments

This notebook demos how to collate results from multiple experiments into one place


Firstly, import the RunCollator class


In [ ]:
from autocast.scripts.utils import RunCollator

In [ ]:
# Directory where multiple runs are stored
OUTPUTS_DIR = "../../isambard-ai/outputs"

# Parameters to extract from the config files
params = {
    "processor_model": "model.processor._target_",
    "processor_hidden_channels": "model.processor.hidden_channels",
}

collator = RunCollator(config_params=params, outputs_dir=OUTPUTS_DIR)

# Set dave csv to true to save the collated results as a csv file in the current directory
df, metadata = collator.collate(save_csv=False, include_metadata_dataframes=True)

In [ ]:
list(metadata.values())[0]

In [ ]:
import pandas as pd

df_lookup = pd.read_excel(
    "~/ati/AutoEmulate - Documents/autocast_results/2026-02-19_autocast.xlsx"
)

In [ ]:
df_lookup

In [ ]:
processor_params_raw = df_lookup["Processor params"]
processor_params_num = pd.to_numeric(processor_params_raw, errors="coerce")

# Fallback for values like "8 channels" or "12.0 params"
missing_num = processor_params_num.isna()
if missing_num.any():
    extracted = pd.to_numeric(
        processor_params_raw.astype(str).str.extract(r"([-+]?\d*\.?\d+)")[0],
        errors="coerce",
    )
    processor_params_num = processor_params_num.where(~missing_num, extracted)

df_lookup["processor_lt_10"] = processor_params_num < 10
df_lookup["small"] = df_lookup["processor_lt_10"]

res = df.merge(df_lookup, left_on="run_name", right_on="Run Name", how="right")


In [ ]:
out_cols = [
    "Short name",
    "small",
    "overall_vrmse",
    "vrmse_0-1",
    "vrmse_6-12",
    "vrmse_13-30",
    "vrmse_31-99",
    "overall_coverage",
    "coverage_0-1",
    "coverage_6-12",
    "coverage_13-30",
    "coverage_31-99",
]

In [ ]:
res[out_cols].round(5)

In [ ]:
import matplotlib.pyplot as plt

performance_cols = ["overall_vrmse", "vrmse_0-1", "vrmse_6-12", "vrmse_13-30", "vrmse_31-99"]
coverage_cols = ["overall_coverage", "coverage_0-1", "coverage_6-12", "coverage_13-30", "coverage_31-99"]

# Consistent split logic across all plots
if "processor_lt_10" in res.columns:
    split_bool = res["processor_lt_10"]
else:
    params_num = pd.to_numeric(res.get("Processor params"), errors="coerce")
    missing_num = params_num.isna()
    if missing_num.any():
        extracted = pd.to_numeric(
            res.get("Processor params").astype(str).str.extract(r"([-+]?\d*\.?\d+)")[0],
            errors="coerce",
        )
        params_num = params_num.where(~missing_num, extracted)
    split_bool = params_num < 10

res_split = res.assign(_is_small=split_bool)
lt_10 = res_split[res_split["_is_small"] == True]
gt_10 = res_split[res_split["_is_small"] == False]

fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)

for col_idx, (subset, title_suffix) in enumerate([(lt_10, "< 10"), (gt_10, "> 10")]):
    perf_mean = subset[performance_cols].mean().sort_index()
    cov_mean = subset[coverage_cols].mean().sort_index()

    perf_mean.plot(kind="bar", ax=axes[0, col_idx])
    cov_mean.plot(kind="bar", ax=axes[1, col_idx])

    axes[0, col_idx].set_title(f"Performance metrics ({title_suffix})")
    axes[1, col_idx].set_title(f"Coverage metrics ({title_suffix})")
    axes[0, col_idx].set_ylabel("Mean VRMSE")
    axes[1, col_idx].set_ylabel("Mean Coverage")
    axes[0, col_idx].tick_params(axis="x", rotation=45)
    axes[1, col_idx].tick_params(axis="x", rotation=45)

fig.suptitle("Processor split using processor_lt_10", fontsize=14)
plt.show()

In [ ]:
# Scatter comparison: one row per metric suffix, paired <10 and >10 panels

model_col = "Short name" if "Short name" in res.columns else ("processor_model" if "processor_model" in res.columns else "run_name")
allowed_models = set(res[model_col].dropna().astype(str).unique())

pair_specs = [
    ("overall", "overall_vrmse", "overall_coverage"),
    ("0-1", "vrmse_0-1", "coverage_0-1"),
    ("6-12", "vrmse_6-12", "coverage_6-12"),
    ("13-30", "vrmse_13-30", "coverage_13-30"),
    ("31-99", "vrmse_31-99", "coverage_31-99"),
]

# Prefer explicit boolean split if present; fallback to numeric split column
if "processor_lt_10" in res.columns:
    res_split = res.assign(_is_small=res["processor_lt_10"])
else:
    if "_split_value" not in res_split.columns:
        res_split = res.assign(_split_value=pd.to_numeric(res[split_col], errors="coerce"))
    res_split = res_split.assign(_is_small=res_split["_split_value"] < 10)

records = []
for _, row in res_split.iterrows():
    is_small = row.get("_is_small")
    if pd.isna(is_small):
        continue

    model_name = str(row.get(model_col))
    if model_name not in allowed_models:
        continue

    split_group = "< 10" if bool(is_small) else "> 10"

    for metric_label, vrmse_col, coverage_col in pair_specs:
        vrmse_val = pd.to_numeric(pd.Series([row.get(vrmse_col)]), errors="coerce").iloc[0]
        coverage_val = pd.to_numeric(pd.Series([row.get(coverage_col)]), errors="coerce").iloc[0]
        if pd.isna(vrmse_val) or pd.isna(coverage_val):
            continue
        records.append(
            {
                "model": model_name,
                "split": split_group,
                "metric": metric_label,
                "vrmse": vrmse_val,
                "coverage": coverage_val,
            }
        )

scatter_df = pd.DataFrame(records)

metrics_order = [m[0] for m in pair_specs]
splits = ["< 10", "> 10"]

fig, axes = plt.subplots(
    nrows=len(metrics_order),
    ncols=len(splits),
    figsize=(16, 3.6 * len(metrics_order)),
    constrained_layout=True,
)

if len(metrics_order) == 1:
    axes = axes.reshape(1, -1)

model_order = sorted(scatter_df["model"].unique()) if not scatter_df.empty else []
color_map = {
    model_name: plt.cm.tab20(i % 20)
    for i, model_name in enumerate(model_order)
}

for row_idx, metric_name in enumerate(metrics_order):
    for col_idx, split_group in enumerate(splits):
        ax = axes[row_idx, col_idx]
        panel_data = scatter_df[
            (scatter_df["metric"] == metric_name) & (scatter_df["split"] == split_group)
        ]

        if panel_data.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center")
            ax.set_title(f"{metric_name}: coverage vs vrmse ({split_group})")
            ax.set_axis_off()
            continue

        for model_name, model_data in panel_data.groupby("model"):
            ax.scatter(
                model_data["vrmse"],
                model_data["coverage"],
                s=70,
                alpha=0.85,
                color=color_map.get(model_name),
                label=model_name,
            )

        ax.set_title(f"{metric_name}: coverage vs vrmse ({split_group})")
        ax.set_xlabel("VRMSE")
        ax.set_ylabel("Coverage")
        ax.grid(True, alpha=0.3)

handles, labels = [], []
for ax in axes.flat:
    h, l = ax.get_legend_handles_labels()
    handles.extend(h)
    labels.extend(l)

seen = set()
unique_handles = []
unique_labels = []
for h, l in zip(handles, labels):
    if l not in seen:
        seen.add(l)
        unique_handles.append(h)
        unique_labels.append(l)

if unique_handles:
    fig.legend(
        unique_handles,
        unique_labels,
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        title=model_col,
    )

fig.suptitle("Coverage vs VRMSE by metric suffix (processor split from processor_lt_10)", fontsize=14)
plt.show()

By Default, all the performance metrics are collected as well as any run parameters specified in the params dictionary.
If a parameter can't be found for that run, then its filled with a N/A.


In [ ]:
# Row-wise combined scatter: one row per metric pair, small/large overlaid via marker
from matplotlib.lines import Line2D

if "scatter_df" not in globals() or scatter_df.empty:
    raise ValueError("scatter_df is empty. Run the scatter preparation cell first.")

combined = scatter_df[scatter_df["split"].isin(["< 10", "> 10"])].copy()
metrics_order = ["overall", "0-1", "6-12", "13-30", "31-99"]

model_order = sorted(combined["model"].unique())
model_colors = {model_name: plt.cm.tab20(i % 20) for i, model_name in enumerate(model_order)}
split_markers = {"< 10": "o", "> 10": "^"}

y_max_coverage = 0.4

fig, axes = plt.subplots(
    nrows=len(metrics_order),
    ncols=1,
    figsize=(14, 3.5 * len(metrics_order)),
    constrained_layout=True,
)

if len(metrics_order) == 1:
    axes = [axes]

model_handles = [
    Line2D([0], [0], marker="o", linestyle="", color=model_colors[m], markersize=7, label=m)
    for m in model_order
]
split_handles = [
    Line2D([0], [0], marker=mk, linestyle="", color="black", markersize=7, label=grp)
    for grp, mk in split_markers.items()
]

for row_idx, metric_name in enumerate(metrics_order):
    ax = axes[row_idx]
    panel_data = combined[combined["metric"] == metric_name].copy()

    if panel_data.empty:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_title(f"{metric_name}: coverage vs vrmse (small+large)")
        ax.set_axis_off()
        continue

    positive_vrmse = panel_data.loc[panel_data["vrmse"] > 0, "vrmse"]
    if positive_vrmse.empty:
        vrmse_floor = 1e-8
    else:
        vrmse_floor = positive_vrmse.min() * 0.5
    panel_data["vrmse_plot"] = panel_data["vrmse"].clip(lower=vrmse_floor)

    for (model_name, split_group), group_df in panel_data.groupby(["model", "split"]):
        ax.scatter(
            group_df["vrmse_plot"],
            group_df["coverage"],
            color=model_colors[model_name],
            marker=split_markers[split_group],
            s=70,
            alpha=0.85,
        )

    ax.set_xscale("log")
    x_min = panel_data["vrmse_plot"].min() * 0.8
    x_max = panel_data["vrmse_plot"].max() * 1.25
    ax.set_xlim(left=x_min, right=x_max)
    ax.set_ylim(bottom=0, top=y_max_coverage)

    n_zero_or_negative = int((panel_data["vrmse"] <= 0).sum())

    ax.set_title(f"{metric_name}: coverage vs vrmse (small+large)")
    ax.set_xlabel("VRMSE (log scale)")
    ax.set_ylabel("Coverage")
    ax.grid(True, alpha=0.3, which="both")

    if n_zero_or_negative > 0:
        ax.text(
            0.01,
            0.01,
            f"Adjusted non-positive VRMSE points: {n_zero_or_negative}",
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=8,
        )

    legend_models = ax.legend(
        handles=model_handles,
        title=model_col,
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
        fontsize=8,
    )
    ax.add_artist(legend_models)
    ax.legend(
        handles=split_handles,
        title="Processor split",
        loc="lower left",
        bbox_to_anchor=(1.01, 0.0),
        fontsize=8,
    )

fig.suptitle("Coverage vs VRMSE by metric (rows), small/large combined", fontsize=14)
plt.show()

In [ ]:
res[["Short name", "Processor params", "processor_lt_10", "small"]].drop_duplicates().sort_values("Short name")

In [ ]:
# Timing comparisons across models (training, inference, test, rollout) with small/large split
from matplotlib.lines import Line2D

model_col = "Short name" if "Short name" in res.columns else "run_name"

if "processor_lt_10" in res.columns:
    split_bool = res["processor_lt_10"]
else:
    params_num = pd.to_numeric(res.get("Processor params"), errors="coerce")
    split_bool = params_num < 10

run_map = (
    res.assign(processor_lt_10=split_bool)
    [["run_name", model_col, "processor_lt_10"]]
    .dropna(subset=["run_name", model_col, "processor_lt_10"])
    .drop_duplicates(subset=["run_name"])
)

meta_all = pd.concat(metadata.values(), ignore_index=True).copy()
meta_all["value"] = pd.to_numeric(meta_all["value"], errors="coerce")

# Build timing table by stage (seconds)
stage_frames = []

# Training time from lookup sheet
if "Train time mins" in df_lookup.columns:
    train_seconds = pd.to_numeric(df_lookup["Train time mins"], errors="coerce") * 60.0
    training_df = pd.DataFrame(
        {
            "run_name": df_lookup["Run Name"],
            "stage": "training",
            "seconds": train_seconds,
        }
    )
    stage_frames.append(training_df)

# Inference: mean time per batch on test loader
inference_df = meta_all[
    (meta_all["loader"] == "test_dataloader")
    & (meta_all["metric"] == "per_batch_mean_s")
][["run_name", "value"]].copy()
inference_df = inference_df.rename(columns={"value": "seconds"})
inference_df["stage"] = "inference"
stage_frames.append(inference_df)

# Test: total eval time on test loader
test_df = meta_all[
    (meta_all["loader"] == "test_dataloader")
    & (meta_all["metric"] == "total_s")
][["run_name", "value"]].copy()
test_df = test_df.rename(columns={"value": "seconds"})
test_df["stage"] = "test"
stage_frames.append(test_df)

# Rollout: total eval time on rollout loader
rollout_df = meta_all[
    (meta_all["loader"] == "rollout_dataloader")
    & (meta_all["metric"] == "total_s")
][["run_name", "value"]].copy()
rollout_df = rollout_df.rename(columns={"value": "seconds"})
rollout_df["stage"] = "rollout"
stage_frames.append(rollout_df)

timing_long = pd.concat(stage_frames, ignore_index=True)
timing_long = timing_long.dropna(subset=["run_name", "seconds", "stage"])

# Aggregate in case a run has repeated timing rows
timing_long = (
    timing_long.groupby(["run_name", "stage"], as_index=False)["seconds"].mean()
)

timing_plot = timing_long.merge(run_map, on="run_name", how="inner")
timing_plot["split"] = timing_plot["processor_lt_10"].map({True: "< 10", False: "> 10"})

stage_order = ["training", "inference", "test", "rollout"]
available_stages = [stage for stage in stage_order if stage in timing_plot["stage"].unique()]

if not available_stages:
    raise ValueError("No timing stages found for plotting.")

model_order = sorted(timing_plot[model_col].astype(str).unique())
model_to_y = {model_name: idx for idx, model_name in enumerate(model_order)}
model_colors = {model_name: plt.cm.tab20(idx % 20) for idx, model_name in enumerate(model_order)}
split_markers = {"< 10": "o", "> 10": "^"}

fig, axes = plt.subplots(
    nrows=len(available_stages),
    ncols=1,
    figsize=(15, 2.8 * len(available_stages)),
    constrained_layout=True,
)
if len(available_stages) == 1:
    axes = [axes]

for axis_idx, stage_name in enumerate(available_stages):
    axis = axes[axis_idx]
    stage_data = timing_plot[timing_plot["stage"] == stage_name].copy()

    if stage_data.empty:
        axis.text(0.5, 0.5, "No data", ha="center", va="center")
        axis.set_title(f"{stage_name} time")
        axis.set_axis_off()
        continue

    stage_data["model_name"] = stage_data[model_col].astype(str)
    stage_data["y_pos"] = stage_data["model_name"].map(model_to_y)

    positive_seconds = stage_data.loc[stage_data["seconds"] > 0, "seconds"]
    floor_seconds = positive_seconds.min() * 0.5 if not positive_seconds.empty else 1e-8
    stage_data["seconds_plot"] = stage_data["seconds"].clip(lower=floor_seconds)

    for split_name, split_subset in stage_data.groupby("split"):
        axis.scatter(
            split_subset["seconds_plot"],
            split_subset["y_pos"],
            marker=split_markers.get(split_name, "o"),
            s=80,
            alpha=0.9,
            c=[model_colors[model_name] for model_name in split_subset["model_name"]],
        )

    axis.set_xscale("log")
    axis.set_yticks(range(len(model_order)))
    axis.set_yticklabels(model_order)
    axis.set_xlabel("Time (seconds, log scale)")
    axis.set_title(f"{stage_name.capitalize()} timing by model")
    axis.grid(True, alpha=0.3, which="both", axis="x")

# Full legends on the right
model_handles = [
    Line2D([0], [0], marker="o", linestyle="", color=model_colors[m], markersize=7, label=m)
    for m in model_order
]
split_handles = [
    Line2D([0], [0], marker=marker, linestyle="", color="black", markersize=7, label=split_name)
    for split_name, marker in split_markers.items()
]

legend_models = fig.legend(
    handles=model_handles,
    title=model_col,
    loc="center left",
    bbox_to_anchor=(1.01, 0.65),
    fontsize=8,
)
fig.add_artist(legend_models)
fig.legend(
    handles=split_handles,
    title="Processor split",
    loc="center left",
    bbox_to_anchor=(1.01, 0.35),
    fontsize=8,
)

fig.suptitle("Timing comparison across models (small/large split)", fontsize=14)
plt.show()

In [ ]:
# Additional plots:
# 1) Number of params vs timing (training / inference / test / rollout)
# 2) VRMSE vs number of params (one row per metric)
from matplotlib.lines import Line2D

model_col = "Short name" if "Short name" in res.columns else "run_name"

# Robust numeric parameter extraction
params_raw = res.get("Processor params")
params_num = pd.to_numeric(params_raw, errors="coerce")
missing_num = params_num.isna()
if missing_num.any():
    extracted = pd.to_numeric(
        params_raw.astype(str).str.extract(r"([-+]?\d*\.?\d+)")[0],
        errors="coerce",
    )
    params_num = params_num.where(~missing_num, extracted)

# Consistent small/large split
if "processor_lt_10" in res.columns:
    split_bool = res["processor_lt_10"]
else:
    split_bool = params_num < 10

run_map = (
    res.assign(processor_lt_10=split_bool, n_params=params_num)
    [["run_name", model_col, "processor_lt_10", "n_params"]]
    .dropna(subset=["run_name", model_col, "processor_lt_10", "n_params"])
    .drop_duplicates(subset=["run_name"])
)

model_order = sorted(run_map[model_col].astype(str).unique())
model_colors = {model_name: plt.cm.tab20(idx % 20) for idx, model_name in enumerate(model_order)}
split_markers = {"< 10": "o", "> 10": "^"}

# ---- Build timing long table (seconds) ----
meta_all = pd.concat(metadata.values(), ignore_index=True).copy()
meta_all["value"] = pd.to_numeric(meta_all["value"], errors="coerce")

stage_frames = []

if "Train time mins" in df_lookup.columns:
    train_seconds = pd.to_numeric(df_lookup["Train time mins"], errors="coerce") * 60.0
    training_df = pd.DataFrame(
        {
            "run_name": df_lookup["Run Name"],
            "stage": "training",
            "seconds": train_seconds,
        }
    )
    stage_frames.append(training_df)

inference_df = meta_all[
    (meta_all["loader"] == "test_dataloader")
    & (meta_all["metric"] == "per_batch_mean_s")
][["run_name", "value"]].copy()
inference_df = inference_df.rename(columns={"value": "seconds"})
inference_df["stage"] = "inference"
stage_frames.append(inference_df)

test_df = meta_all[
    (meta_all["loader"] == "test_dataloader")
    & (meta_all["metric"] == "total_s")
][["run_name", "value"]].copy()
test_df = test_df.rename(columns={"value": "seconds"})
test_df["stage"] = "test"
stage_frames.append(test_df)

rollout_df = meta_all[
    (meta_all["loader"] == "rollout_dataloader")
    & (meta_all["metric"] == "total_s")
][["run_name", "value"]].copy()
rollout_df = rollout_df.rename(columns={"value": "seconds"})
rollout_df["stage"] = "rollout"
stage_frames.append(rollout_df)

timing_long = pd.concat(stage_frames, ignore_index=True)
timing_long = timing_long.dropna(subset=["run_name", "seconds", "stage"])
timing_long = timing_long.groupby(["run_name", "stage"], as_index=False)["seconds"].mean()

timing_plot = timing_long.merge(run_map, on="run_name", how="inner")
timing_plot["split"] = timing_plot["processor_lt_10"].map({True: "< 10", False: "> 10"})

stage_order = ["training", "inference", "test", "rollout"]
available_stages = [s for s in stage_order if s in timing_plot["stage"].unique()]

fig, axes = plt.subplots(
    nrows=max(1, len(available_stages)),
    ncols=1,
    figsize=(14, 3.0 * max(1, len(available_stages))),
    constrained_layout=True,
)
if len(available_stages) == 1:
    axes = [axes]

for idx, stage_name in enumerate(available_stages):
    ax = axes[idx]
    stage_data = timing_plot[timing_plot["stage"] == stage_name].copy()

    for split_name, split_subset in stage_data.groupby("split"):
        ax.scatter(
            split_subset["n_params"],
            split_subset["seconds"],
            marker=split_markers.get(split_name, "o"),
            s=80,
            alpha=0.9,
            c=[model_colors[str(model_name)] for model_name in split_subset[model_col].astype(str)],
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Number of params (log scale)")
    ax.set_ylabel("Time (seconds, log scale)")
    ax.set_title(f"{stage_name.capitalize()}: params vs time")
    ax.grid(True, alpha=0.3, which="both")

model_handles = [
    Line2D([0], [0], marker="o", linestyle="", color=model_colors[m], markersize=7, label=m)
    for m in model_order
]
split_handles = [
    Line2D([0], [0], marker=mk, linestyle="", color="black", markersize=7, label=grp)
    for grp, mk in split_markers.items()
]

legend_models = fig.legend(
    handles=model_handles,
    title=model_col,
    loc="center left",
    bbox_to_anchor=(1.01, 0.65),
    fontsize=8,
)
fig.add_artist(legend_models)
fig.legend(
    handles=split_handles,
    title="Processor split",
    loc="center left",
    bbox_to_anchor=(1.01, 0.35),
    fontsize=8,
)
fig.suptitle("Params vs timing by stage (small/large split)", fontsize=14)
plt.show()

# ---- VRMSE vs params (one row per metric) ----
vrmse_specs = [
    ("overall", "overall_vrmse"),
    ("0-1", "vrmse_0-1"),
    ("6-12", "vrmse_6-12"),
    ("13-30", "vrmse_13-30"),
    ("31-99", "vrmse_31-99"),
]

vrmse_cols = [col for _, col in vrmse_specs]
vrmse_base = res.assign(n_params=params_num, processor_lt_10=split_bool)
vrmse_base = vrmse_base[["run_name", model_col, "n_params", "processor_lt_10", *vrmse_cols]].copy()
vrmse_base = vrmse_base.dropna(subset=["n_params", model_col])

vrmse_long = vrmse_base.melt(
    id_vars=["run_name", model_col, "n_params", "processor_lt_10"],
    value_vars=vrmse_cols,
    var_name="metric_col",
    value_name="vrmse",
)
vrmse_long["vrmse"] = pd.to_numeric(vrmse_long["vrmse"], errors="coerce")
vrmse_long = vrmse_long.dropna(subset=["vrmse"])
vrmse_long = vrmse_long[vrmse_long["vrmse"] > 0]

col_to_label = {col: label for label, col in vrmse_specs}
vrmse_long["metric_label"] = vrmse_long["metric_col"].map(col_to_label)
vrmse_long["split"] = vrmse_long["processor_lt_10"].map({True: "< 10", False: "> 10"})

metric_rows = [label for label, _ in vrmse_specs]

fig, axes = plt.subplots(
    nrows=len(metric_rows),
    ncols=1,
    figsize=(14, 2.9 * len(metric_rows)),
    constrained_layout=True,
)
if len(metric_rows) == 1:
    axes = [axes]

for row_idx, metric_label in enumerate(metric_rows):
    ax = axes[row_idx]
    metric_data = vrmse_long[vrmse_long["metric_label"] == metric_label].copy()

    if metric_data.empty:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_title(f"VRMSE {metric_label}: params comparison")
        ax.set_axis_off()
        continue

    for split_name, split_subset in metric_data.groupby("split"):
        ax.scatter(
            split_subset["n_params"],
            split_subset["vrmse"],
            marker=split_markers.get(split_name, "o"),
            s=80,
            alpha=0.9,
            c=[model_colors[str(model_name)] for model_name in split_subset[model_col].astype(str)],
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Number of params (log scale)")
    ax.set_ylabel("VRMSE (log scale)")
    ax.set_title(f"VRMSE {metric_label}: params vs VRMSE")
    ax.grid(True, alpha=0.3, which="both")

legend_models = fig.legend(
    handles=model_handles,
    title=model_col,
    loc="center left",
    bbox_to_anchor=(1.01, 0.65),
    fontsize=8,
)
fig.add_artist(legend_models)
fig.legend(
    handles=split_handles,
    title="Processor split",
    loc="center left",
    bbox_to_anchor=(1.01, 0.35),
    fontsize=8,
)
fig.suptitle("VRMSE vs number of params (rows by metric, small/large split)", fontsize=14)
plt.show()

In [ ]:
# Inspect potential batch/ensemble/time fields for per-sample normalization
print('df columns with batch/ensemble/sample/time keywords:')
keys = ['batch', 'ensemble', 'sample', 'time', 'rollout', 'inferen', 'test', 'train']
for c in sorted([col for col in df.columns if any(k in col.lower() for k in keys)]):
    print('-', c)

import pandas as pd
meta_all = pd.concat(metadata.values(), ignore_index=True)
print('\nmetadata metrics:')
for m in sorted(meta_all['metric'].dropna().astype(str).unique()):
    print('-', m)

print('\nmetadata loaders:')
for l in sorted(meta_all['loader'].dropna().astype(str).unique()):
    print('-', l)

In [ ]:
# Per-sample normalized timing comparison (new plots; does not modify existing ones)
# Normalization uses batch size and ensemble members from run configs when available.
from matplotlib.lines import Line2D
from autocast.scripts.utils import RunCollator

# Enrich run-level config fields needed for normalization
norm_params = {
    "train_batch_size": "datamodule.batch_size",
    "eval_batch_size": "eval.batch_size",
    "train_n_members": "model.n_members",
    "eval_n_members": "eval.n_members",
    "n_epochs": "trainer.max_epochs",
}
collator_norm = RunCollator(config_params=norm_params, outputs_dir=OUTPUTS_DIR)
df_norm = collator_norm.collate(save_csv=False, include_metadata_dataframes=False)

model_col = "Short name" if "Short name" in res.columns else "run_name"

# Base run map from res + lookup-derived training time/epochs
base_map = (
    res[["run_name", model_col, "processor_lt_10"]]
    .dropna(subset=["run_name", model_col, "processor_lt_10"])
    .drop_duplicates(subset=["run_name"])
)

lookup_times = pd.DataFrame(
    {
        "run_name": df_lookup["Run Name"],
        "train_time_s": pd.to_numeric(df_lookup["Train time mins"], errors="coerce") * 60.0,
        "lookup_epochs": pd.to_numeric(df_lookup["N epochs"], errors="coerce"),
    }
)

cfg_map = df_norm[["run_name", *norm_params.keys()]].copy()
for col in ["train_batch_size", "eval_batch_size", "train_n_members", "eval_n_members", "n_epochs"]:
    cfg_map[col] = pd.to_numeric(cfg_map[col], errors="coerce")

runs = (
    base_map
    .merge(lookup_times, on="run_name", how="left")
    .merge(cfg_map, on="run_name", how="left")
)

# Fill defaults where config is missing
runs["train_batch_size"] = runs["train_batch_size"].fillna(1.0)
runs["eval_batch_size"] = runs["eval_batch_size"].fillna(runs["train_batch_size"])
runs["train_n_members"] = runs["train_n_members"].fillna(1.0)
runs["eval_n_members"] = runs["eval_n_members"].fillna(runs["train_n_members"])
runs["n_epochs_eff"] = runs["n_epochs"].fillna(runs["lookup_epochs"]).fillna(1.0)

# Pull per-batch eval timings from metadata; fallback total_s/batch_count if needed
meta_all = pd.concat(metadata.values(), ignore_index=True).copy()
meta_all["value"] = pd.to_numeric(meta_all["value"], errors="coerce")

def loader_per_batch_seconds(loader_name: str) -> pd.DataFrame:
    pb = meta_all[(meta_all["loader"] == loader_name) & (meta_all["metric"] == "per_batch_mean_s")][["run_name", "value"]].copy()
    if not pb.empty:
        pb = pb.rename(columns={"value": "per_batch_s"})
        return pb.groupby("run_name", as_index=False)["per_batch_s"].mean()

    total = meta_all[(meta_all["loader"] == loader_name) & (meta_all["metric"] == "total_s")][["run_name", "value"]].rename(columns={"value": "total_s"})
    count = meta_all[(meta_all["loader"] == loader_name) & (meta_all["metric"] == "batch_count")][["run_name", "value"]].rename(columns={"value": "batch_count"})
    merged = total.merge(count, on="run_name", how="inner")
    merged["per_batch_s"] = merged["total_s"] / merged["batch_count"].replace(0, pd.NA)
    return merged[["run_name", "per_batch_s"]].dropna(subset=["per_batch_s"])

infer_pb = loader_per_batch_seconds("test_dataloader").rename(columns={"per_batch_s": "infer_per_batch_s"})
roll_pb = loader_per_batch_seconds("rollout_dataloader").rename(columns={"per_batch_s": "rollout_per_batch_s"})

runs = runs.merge(infer_pb, on="run_name", how="left").merge(roll_pb, on="run_name", how="left")

# Per-sample normalization
# Training uses per-epoch time normalized by train batch and train ensemble size.
runs["train_per_sample_s"] = (
    (runs["train_time_s"] / runs["n_epochs_eff"].replace(0, pd.NA))
    / runs["train_batch_size"].replace(0, pd.NA)
    / runs["train_n_members"].replace(0, pd.NA)
)

# Inference/rollout use per-batch metadata normalized by eval batch and eval ensemble size.
runs["inference_per_sample_s"] = (
    runs["infer_per_batch_s"]
    / runs["eval_batch_size"].replace(0, pd.NA)
    / runs["eval_n_members"].replace(0, pd.NA)
)
runs["rollout_per_sample_s"] = (
    runs["rollout_per_batch_s"]
    / runs["eval_batch_size"].replace(0, pd.NA)
    / runs["eval_n_members"].replace(0, pd.NA)
)

runs["split"] = runs["processor_lt_10"].map({True: "< 10", False: "> 10"})

stage_specs = [
    ("training", "train_per_sample_s", "Training sec/sample (per-epoch normalized)"),
    ("inference", "inference_per_sample_s", "Inference sec/sample"),
    ("rollout", "rollout_per_sample_s", "Rollout sec/sample"),
]

model_order = sorted(runs[model_col].astype(str).unique())
model_to_y = {m: i for i, m in enumerate(model_order)}
model_colors = {m: plt.cm.tab20(i % 20) for i, m in enumerate(model_order)}
split_markers = {"< 10": "o", "> 10": "^"}

fig, axes = plt.subplots(nrows=len(stage_specs), ncols=1, figsize=(15, 3.0 * len(stage_specs)), constrained_layout=True)
if len(stage_specs) == 1:
    axes = [axes]

for i, (stage_label, value_col, x_label) in enumerate(stage_specs):
    ax = axes[i]
    plot_df = runs[[model_col, "split", value_col]].copy()
    plot_df = plot_df.dropna(subset=[value_col])
    plot_df = plot_df[plot_df[value_col] > 0]

    if plot_df.empty:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_title(f"{stage_label.capitalize()} per-sample timing")
        ax.set_axis_off()
        continue

    plot_df["model_name"] = plot_df[model_col].astype(str)
    plot_df["y_pos"] = plot_df["model_name"].map(model_to_y)

    for split_name, subset in plot_df.groupby("split"):
        ax.scatter(
            subset[value_col],
            subset["y_pos"],
            marker=split_markers.get(split_name, "o"),
            s=85,
            alpha=0.9,
            c=[model_colors[m] for m in subset["model_name"]],
        )

    ax.set_xscale("log")
    ax.set_yticks(range(len(model_order)))
    ax.set_yticklabels(model_order)
    ax.set_xlabel(x_label + " (log scale)")
    ax.set_title(f"{stage_label.capitalize()} per-sample timing by model")
    ax.grid(True, alpha=0.3, which="both", axis="x")

model_handles = [
    Line2D([0], [0], marker="o", linestyle="", color=model_colors[m], markersize=7, label=m)
    for m in model_order
]
split_handles = [
    Line2D([0], [0], marker=mk, linestyle="", color="black", markersize=7, label=grp)
    for grp, mk in split_markers.items()
]

legend_models = fig.legend(handles=model_handles, title=model_col, loc="center left", bbox_to_anchor=(1.01, 0.65), fontsize=8)
fig.add_artist(legend_models)
fig.legend(handles=split_handles, title="Processor split", loc="center left", bbox_to_anchor=(1.01, 0.35), fontsize=8)

fig.suptitle("Per-sample timing comparison (normalized for batch size and ensemble)", fontsize=14)
plt.show()

# Quick table for transparency of normalization inputs
display(
    runs[[model_col, "train_batch_size", "eval_batch_size", "train_n_members", "eval_n_members", "n_epochs_eff", "train_per_sample_s", "inference_per_sample_s", "rollout_per_sample_s"]]
    .drop_duplicates(subset=[model_col])
    .sort_values(model_col)
)

In [ ]:
# Focused debug table for diffusion/flow-matching size pairs
focus_models = ["Diffusion", "Diffusion_small", "Flow_matching", "Flow_matching_small"]
cols = [
    model_col,
    "train_batch_size",
    "eval_batch_size",
    "train_n_members",
    "eval_n_members",
    "n_epochs_eff",
    "train_per_sample_s",
    "infer_per_batch_s",
    "rollout_per_batch_s",
    "inference_per_sample_s",
    "rollout_per_sample_s",
]
focus = runs[runs[model_col].isin(focus_models)][cols].drop_duplicates(subset=[model_col]).sort_values(model_col)
display(focus)

# Ratios small/large
pairs = [("Diffusion_small", "Diffusion"), ("Flow_matching_small", "Flow_matching")]
for small_name, large_name in pairs:
    s = focus[focus[model_col] == small_name].iloc[0]
    l = focus[focus[model_col] == large_name].iloc[0]
    print(f"\n{small_name} vs {large_name}")
    print("  inference per-sample ratio:", float(s["inference_per_sample_s"] / l["inference_per_sample_s"]))
    print("  rollout per-sample ratio:", float(s["rollout_per_sample_s"] / l["rollout_per_sample_s"]))
    print("  infer per-batch ratio:", float(s["infer_per_batch_s"] / l["infer_per_batch_s"]))
    print("  rollout per-batch ratio:", float(s["rollout_per_batch_s"] / l["rollout_per_batch_s"]))